### US ACCIDENTS DCDM notebook (data cleaning and management for subsequent models being used on it)
    - this is a test notebook, it will contain cleaning code and potential OOP segements to re-use across projects 

Steps of the Cleaning pipeline:
1. confirm new starting point status - source 2, row counts, column listing, id uniqueness, source value duplicates
2. duplicates
    - rerun duplicate check - exclude ID 
    - check source and start_year within the duplicate groups 
    - drop: 'keep_first = True' - prior to any split or copies will land on either side 
3. leakage column removal: 
    - use time_delta once to validate the target - severe accidents should last longer?
    - then drop: end_time, end_lat, end_lng, distance(mi), description
4. remove:
    - bump, give_way, start_month, roundabout, source
    - keep railway, no_exist, traffic_calming = allow for L1 regularisation decide (feature importance) 
5. keep until event:
    - drop raw_street, raw_weather conditions, four light columns, start_time, weather_timestamp
    - keep airport_code until the missing cehck - then remove 
    - keep ID until after spilt, then remove from x_input feature matrix
6. remove impossible values with NAN replacement
7. build final feature matrix: severe (severity in 3,4 = target) - decide if you want to combine or no 
    - is_night = (light_score = 4) - else 0
    - constraint count - how many traffic_signals, crossings, stop, amenities, stations are True
    - determine if traffic_signal, stop, constraint count are kept + drop crossing, amentiy, station 
    - encode - ordinal for trees, one hot encoding for logistic regression - state gives you 49 columns if this is the case 
8. missing values: test if missing predicts severity 
    - correlate wind_chill against temp; above 0.95 drop it as a correlated column of noise 
    - decide if precipition with 47% missing values and over 75% are useful 
    - test the airport_code link to missing weather - then drop airport code 
    - drop 1541 missing rows in light_score 
    - cateogrical features - get an unknown categories not a guess or uncertain estimate
9. split and fit:
    - train/val/test - 60/20/20
    - random split - target is stable across srouce 2 years therefore - representative 
    - fit imputation on TRAINING only - apply to validation and test 
    - scaling only if using logitsic regression - TREES DONT NEED IT 
    

Imports

In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
from sklearn.model_selection import train_test_split
import pyarrow.parquet as pq
from sklearn.impute import SimpleImputer
source2 = pq.read_table('../data/source2.parquet').to_pandas()


In [2]:
# create a copy so that you dont work on the origonal 
df = source2.copy()
 

In [3]:
df.columns
df.shape


(3305373, 56)

In [4]:
# duplicates: check duplicates across columns and across source and start_year
columns_to_check = df.columns.drop('id')
duplicates = df[df.duplicated(subset=columns_to_check, keep='first')]


In [5]:
duplicates.shape

(3581, 56)

In [6]:
type(duplicates)
# how many times a value repeats except for first occurance
# Count the frequency of each combination
combination_counts = duplicates[['source', 'start_year']].value_counts(dropna=False)
print(combination_counts)
#3305373 total & 3581 duplicate rows 

source   start_year
Source2  2018          1070
         2019           645
         2021           605
         2020           438
         2017           381
         2022           323
         2016           119
Name: count, dtype: int64


In [7]:
# drop the duplicated rows;
df_cleaned = df.drop_duplicates(subset=columns_to_check, keep='first')

In [8]:
df_cleaned.shape # should be 3301792

(3301792, 56)

3. Leakage columns:
    - look at severity across median time_delta values
4. columns identified as redundant

In [9]:
df_cleaned.groupby('severity')['time_delta'].apply(lambda x: x.median())
df_cleaned.groupby('severity')['distance(mi)'].apply(lambda x: x.median())
# across both there is no difference aceross 1,2,3 but 4 shows a difference, ths means that severity is not rpedicted via distance or time delta, but something else important in the dataset - remove these but keep others 



severity
1    0.00
2    0.00
3    0.00
4    2.95
Name: distance(mi), dtype: float64

In [10]:
df_cleaned.drop(columns=['time_delta', 'distance(mi)','end_lat', 'end_lng', 'end_time'], inplace = True)


In [11]:
# columns which stage 2 of EDA confirmed were redundant; a measured near-zero effect, too few rows to measure anything, and a column that's now constant.
df_cleaned.drop(columns = ['bump', 'give_way', 'start_month', 'roundabout', 'source'], inplace = True)

5. drop old columns which were used to engineer new features - they are no longer needed

In [12]:
df_cleaned.columns

Index(['id', 'severity', 'start_time', 'start_lat', 'start_lng', 'street',
       'state', 'timezone(us)', 'airport_code', 'weather_timestamp',
       'temperature(f)', 'wind_chill(f)', 'humidity(%)', 'pressure(in)',
       'visibility(mi)', 'wind_direction', 'wind_speed(mph)',
       'precipitation(in)', 'weather_condition', 'amenity', 'crossing',
       'junction', 'no_exit', 'railway', 'station', 'stop', 'traffic_calming',
       'traffic_signal', 'sunrise_sunset', 'civil_twilight',
       'nautical_twilight', 'astronomical_twilight', 'light_score',
       'street_type', 'start_year', 'start_hour', 'start_day', 'is_rain',
       'is_snow', 'is_ice', 'is_thunder', 'is_fog', 'is_obscured', 'is_windy',
       'intensity', 'weather_missing'],
      dtype='str')

In [13]:
df_cleaned.drop(columns = ['street', 'weather_condition', ], inplace = True)

5. Impossible value removal:
    - change all impossible values to NAN

In [14]:
def ranges(df, col, lower, upper):
    # mask changes and puts np.nan in as default, no need to specify 
    return df[col].mask((df[col] > upper) | (df[col] < lower))

# create a dictionary of limits:
limits = {
    'temperature(f)':    (-70, 134),
    'humidity(%)':       (0, 100),
    'pressure(in)':      (20, 32),
    'wind_speed(mph)':   (0, 150),
    'precipitation(in)': (0, 15),
    'start_lat':         (20, 72),
    'start_lng':         (-180, -60),
}

for col, (lower, upper) in limits.items():
    df_cleaned[col] = ranges(df_cleaned, col, lower, upper)

#do wind_chill column clenaing after the loop: it is dependant on temp. ranges
df_cleaned['wind_chill(f)'] = ranges(df_cleaned, 'wind_chill(f)', -80, df_cleaned['temperature(f)'])


In [15]:
df_cleaned.describe(include = 'number')

,severity,start_lat,start_lng,temperature(f),wind_chill(f),humidity(%),pressure(in),visibility(mi),wind_speed(mph),precipitation(in),...,start_day,is_rain,is_snow,is_ice,is_thunder,is_fog,is_obscured,is_windy,intensity,weather_missing
count,3.301792e+06,3.301792e+06,3.301792e+06,3.246166e+06,1.863289e+06,3.242385e+06,3.254106e+06,3.238233e+06,2.944938e+06,1.750464e+06,...,3.301792e+06,3.301792e+06,3.301792e+06,3.301792e+06,3.301792e+06,3.301792e+06,3.301792e+06,3.301792e+06,425197.000000,3.301792e+06
mean,2.332894e+00,3.611683e+01,-9.307625e+01,6.235241e+01,5.656553e+01,6.638588e+01,2.969341e+01,9.096450e+00,7.873518e+00,1.243705e-02,...,2.373577e+00,7.407644e-02,1.643017e-02,1.058819e-03,1.040829e-02,1.344997e-02,1.166367e-02,8.307004e-03,0.498035,1.922895e-02
std,5.019320e-01,4.830007e+00,1.616683e+01,1.861856e+01,2.315084e+01,2.244333e+01,8.216152e-01,2.727664e+00,4.998367e+00,1.504461e-01,...,1.703740e+00,2.618953e-01,1.271229e-01,3.252227e-02,1.014887e-01,1.151915e-01,1.073668e-01,9.076343e-02,0.594612,1.373288e-01
min,1.000000e+00,2.455480e+01,-1.246238e+02,-3.700000e+01,-5.510000e+01,1.000000e+00,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
25%,2.000000e+00,3.319094e+01,-1.049838e+02,5.000000e+01,3.900000e+01,5.000000e+01,2.955000e+01,1.000000e+01,4.600000e+00,0.000000e+00,...,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
50%,2.000000e+00,3.541408e+01,-8.682430e+01,6.440000e+01,6.100000e+01,6.900000e+01,2.992000e+01,1.000000e+01,7.000000e+00,0.000000e+00,...,2.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
75%,3.000000e+00,4.000451e+01,-8.081923e+01,7.600000e+01,7.500000e+01,8.600000e+01,3.007000e+01,1.000000e+01,1.040000e+01,0.000000e+00,...,4.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000,0.000000e+00
max,4.000000e+00,4.900220e+01,-6.755331e+01,1.328000e+02,1.960000e+02,1.000000e+02,3.115000e+01,1.400000e+02,1.450000e+02,1.080000e+01,...,6.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,2.000000,1.000000e+00


7. Building the Final feature matrix
    - severe - build column 
    - build is night column intsead of ordinal light scor e
    - drop light columns 
    - 

In [16]:
df_cleaned['severe'] = df_cleaned['severity'].isin([3,4]).astype(int) # instead of true or false from isin() function output; it does 0 or 1 for numerical boolean values

In [17]:
# light scoring coumns 
df_cleaned.columns
df_cleaned.drop(columns = ['sunrise_sunset', 'civil_twilight', 'nautical_twilight', 'astronomical_twilight'], inplace = True)

In [18]:
# convert light score into 0s and 1s with is_night column 
# drop the na rows:
df_cleaned.dropna(subset=['light_score'], inplace = True)
df_cleaned['is_night'] = (df_cleaned['light_score'] == 4).astype(int)

In [19]:
df_cleaned.groupby(df_cleaned['light_score'].isna())['is_night'].apply(lambda x: x.isin([1.0]).sum())




light_score
False    377966
Name: is_night, dtype: int64

In [20]:
df_cleaned.select_dtypes(include = bool).value_counts(normalize = True, dropna = False)

amenity  crossing  junction  no_exit  railway  station  stop   traffic_calming  traffic_signal
False    False     False     False    False    False    False  False            False             6.562103e-01
                                                                                True              1.102182e-01
         True      False     False    False    False    False  False            True              8.416809e-02
         False     True      False    False    False    False  False            False             4.818691e-02
         True      False     False    False    False    False  False            False             2.389726e-02
                                                                                                      ...     
                   True      True     False    True     False  False            True              3.030071e-07
                   False     True     False    True     True   True             False             3.030071e-07
True     False   

In [21]:
road_feature_cols = ['traffic_signal', 'crossing', 'stop', 'amenity', 'station']
df_cleaned['constraint_count'] = df_cleaned[road_feature_cols].astype(int).sum(axis=1)

In [22]:
df_cleaned.constraint_count.value_counts(normalize = True, dropna = False)*100

constraint_count
0    70.646523
1    16.803591
2    10.772326
3     1.515247
4     0.259556
5     0.002757
Name: proportion, dtype: float64

In [23]:
df_cleaned.drop(columns = ['crossing', 'amenity', 'station'], inplace = True)

7. encoding - final step 
    - we will be building a tree based model - so encoding is required
    - street_type, timezone(us), wind_direction, state, boolean values

In [24]:
from sklearn.preprocessing import OrdinalEncoder
enc = OrdinalEncoder()
# boolean values 

boolean_features = df_cleaned.select_dtypes(include = bool).columns.tolist()
for i in boolean_features:
    df_cleaned[boolean_features] = df_cleaned[boolean_features].astype(int)


In [25]:
cat_cols = ['timezone(us)', 'street_type', 'wind_direction', 'state']
df_cleaned[cat_cols] = enc.fit_transform(df_cleaned[cat_cols])

In [26]:
df_cleaned[cat_cols] # successfully encoded

,timezone(us),street_type,wind_direction,state
index,,,,
0,2.0,2.0,0.0,33.0
1,2.0,3.0,0.0,33.0
2,2.0,5.0,14.0,33.0
3,2.0,2.0,14.0,33.0
4,2.0,3.0,14.0,33.0
...,...,...,...,...
3402757,4.0,3.0,17.0,3.0
3402758,4.0,2.0,13.0,3.0
3402759,4.0,3.0,13.0,3.0


In [27]:
missing_cols = df_cleaned.columns[df_cleaned.isna().sum() > 0]

# 2. Loop through each column to see how missingness affects 'severity'
for col in missing_cols:
    print(f" Analyzing Missingness in: {col} ")
    
    is_missing = df_cleaned[col].isna()
    
    severity_pct = df_cleaned.groupby(is_missing)['severity'].apply(
        lambda x: x.isin([3, 4]).mean() * 100
    )
    
    severity_pct.index = ['Value Present (False)', 'Value Missing (True)']
    print(severity_pct)

 Analyzing Missingness in: airport_code 
Value Present (False)    33.892069
Value Missing (True)     45.293657
Name: severity, dtype: float64
 Analyzing Missingness in: weather_timestamp 
Value Present (False)    33.851228
Value Missing (True)     39.195243
Name: severity, dtype: float64
 Analyzing Missingness in: temperature(f) 
Value Present (False)    33.806510
Value Missing (True)     40.204273
Name: severity, dtype: float64
 Analyzing Missingness in: wind_chill(f) 
Value Present (False)    32.498899
Value Missing (True)     35.746093
Name: severity, dtype: float64
 Analyzing Missingness in: humidity(%) 
Value Present (False)    33.812460
Value Missing (True)     39.471731
Name: severity, dtype: float64
 Analyzing Missingness in: pressure(in) 
Value Present (False)    33.816678
Value Missing (True)     40.576786
Name: severity, dtype: float64
 Analyzing Missingness in: visibility(mi) 
Value Present (False)    33.772851
Value Missing (True)     41.121215
Name: severity, dtype: float

Missingness is not random. All nine weather-related columns show higher severity when missing, from +2.5 to +11.4 points, with airport_code the largest. This supports the hypothesis that unmatched accidents — too remote for a nearby station — are systematically more severe. A single weather_missing indicator is retained rather than per-column flags.

In [28]:
df_cleaned[['wind_chill(f)', 'temperature(f)']].corr()
# drop one column they are highly correlated 
df_cleaned.drop(columns = ['wind_chill(f)'], inplace = True)

In [29]:
# drop precipitation:
df_cleaned.drop(columns = 'precipitation(in)', inplace = True)

In [30]:
df_cleaned.columns

Index(['id', 'severity', 'start_time', 'start_lat', 'start_lng', 'state',
       'timezone(us)', 'airport_code', 'weather_timestamp', 'temperature(f)',
       'humidity(%)', 'pressure(in)', 'visibility(mi)', 'wind_direction',
       'wind_speed(mph)', 'junction', 'no_exit', 'railway', 'stop',
       'traffic_calming', 'traffic_signal', 'light_score', 'street_type',
       'start_year', 'start_hour', 'start_day', 'is_rain', 'is_snow', 'is_ice',
       'is_thunder', 'is_fog', 'is_obscured', 'is_windy', 'intensity',
       'weather_missing', 'severe', 'is_night', 'constraint_count'],
      dtype='str')

In [31]:
# how does airport code link to missing weather
ct = pd.crosstab(
    df_cleaned['airport_code'], 
    df_cleaned['weather_missing'], 
    margins=True, 
    margins_name='Total',
    dropna=False  # Keeps NaN airport codes in the index if present
)

# Rename True/False columns for readability
ct.columns = ['Weather Present', 'Weather Missing', 'Total']
print(ct)
    

              Weather Present  Weather Missing    Total
airport_code                                           
K01M                       23                2       25
K04V                        4                0        4
K04W                      210               10      220
K08D                        1                0        1
K0A9                       55                2       57
...                       ...              ...      ...
KYNG                      358                0      358
KZPH                     2227               64     2291
KZZV                      242                5      247
NaN                         0             6385     6385
Total                 3236868            63385  3300253

[1908 rows x 3 columns]


6,385 missing. Every single accident with no airport_code also has no weather.  all of them.
But it only explains 10% of the gap. 6,385 out of 63,385 missing weather rows. The other 56,999 have an airport code but still no weather reading

In [32]:
# drop airport code:
df_cleaned.drop(columns = 'airport_code', inplace = True)

In [33]:
# cateogrical features change to UNKNOWN variables, NOT MISSING : wind_direction, street_type, state, timezone(us)
df_cleaned[cat_cols].isna().sum()

timezone(us)      0
street_type       0
wind_direction    0
state             0
dtype: int64

In [34]:
# additional column removal due to leakage or redundancy:
df_cleaned.drop(columns = ['start_time', 'weather_timestamp', 'severity', 'light_score', 'intensity'], inplace = True)

In [35]:
df_cleaned.to_parquet('../data/df_cleaned.parquet', engine='fastparquet', index=True)


In [36]:
df_cleaned.columns 

Index(['id', 'start_lat', 'start_lng', 'state', 'timezone(us)',
       'temperature(f)', 'humidity(%)', 'pressure(in)', 'visibility(mi)',
       'wind_direction', 'wind_speed(mph)', 'junction', 'no_exit', 'railway',
       'stop', 'traffic_calming', 'traffic_signal', 'street_type',
       'start_year', 'start_hour', 'start_day', 'is_rain', 'is_snow', 'is_ice',
       'is_thunder', 'is_fog', 'is_obscured', 'is_windy', 'weather_missing',
       'severe', 'is_night', 'constraint_count'],
      dtype='str')

## Split data: 60/20/20
    - impute via medians as results are skewed AFTER the split 
    - fit and transform train 
    - transform val and test

In [37]:
# create x and y df first 
x = df_cleaned.drop(columns=["severe"])
y = df_cleaned["severe"]

x_temp, x_test, y_temp, y_test = train_test_split(x, y, test_size = 0.20, random_state= 42)
x_train, x_val, y_train, y_val = train_test_split(x_temp, y_temp, test_size = 0.25, random_state = 42)


In [38]:
dfs = {'x_train': x_train, 
       'x_test': x_test, 
       'y_train': y_train,
       'y_test': y_test, 
       'x_val': x_val,
       'y_val': y_val}

for name, dataset in dfs.items():
    print(f"shape  of {name}:", {dataset.shape})



shape  of x_train: {(1980151, 31)}
shape  of x_test: {(660051, 31)}
shape  of y_train: {(1980151,)}
shape  of y_test: {(660051,)}
shape  of x_val: {(660051, 31)}
shape  of y_val: {(660051,)}


In [ ]:
# median imputation on columns - they are skewed by mean so median is safer:
# columns imputed on: temperature, humidity, pressure, visibility, wind_speed 
x_train.isna().sum()   
x_val.isna().sum()     
x_test.isna().sum()    
# missing values are consistant across the 5 columns 

# imputer
imputer = SimpleImputer(strategy='median').set_output(transform='pandas')

#columns to transform
trans_cols =['temperature(f)', 'humidity(%)', 'pressure(in)', 'visibility(mi)','wind_speed(mph)']
x_train[trans_cols] = imputer.fit_transform(x_train[trans_cols])


In [40]:
# transform:
x_val[trans_cols] = imputer.transform(x_val[trans_cols])
x_test[trans_cols] = imputer.transform(x_test[trans_cols]) 

In [42]:
# confirm columns are clean
print(x_train.isna().sum().sum(), 
      x_val.isna().sum().sum(), 
      x_test.isna().sum().sum()
      )

0 0 0


In [43]:
import os
os.makedirs('../data/split', exist_ok=True)

x_train.to_parquet('../data/split/x_train.parquet', engine='fastparquet', index=True)
x_val.to_parquet('../data/split/x_val.parquet', engine='fastparquet', index=True)
x_test.to_parquet('../data/split/x_test.parquet', engine='fastparquet', index=True)

y_train.to_frame().to_parquet('../data/split/y_train.parquet', engine='fastparquet', index=True)
y_val.to_frame().to_parquet('../data/split/y_val.parquet', engine='fastparquet', index=True)
y_test.to_frame().to_parquet('../data/split/y_test.parquet', engine='fastparquet', index=True)
